# WRA and Units: using semantic information with QUDT

## Workng with WRA JSON files

Working with WRA JSON files can be handy as it provides a siple yet structured way of handling Wind reosurce-related information. 
For instance, consider this simple WRA file form which we obtain metadata about a measurement point:

In [63]:
import json 

with open("data/simple-wra.json") as json_file:
    meta_data = json.load(json_file)

From here we can get the top-level information like the `organisation` or the `author`:

In [64]:
print('author:',meta_data['author'],'\n' 
      'organization:',meta_data['organisation'])

author: Stephen Holleran 
organization: brightwind


And going deeper into the hierarchy, we can retrieve a specific measurement location, and the type of measurement, e.g. *wind speed*:

In [65]:
location = meta_data['measurement_location'][0]
measurement_point_1 = location['measurement_point'][0]
measurement_point_1['measurement_type_id']

'wind_speed'

Moreover, this measurement point has specific unit of measurement, which are for the moment encoded as this string:

In [66]:
measurement_point_1['logger_measurement_config']['measurement_units_id']

'm/s'

This string is defined and validated in WRA using JSON Schema. All possible units are defined in an enum as follows:

![](img/schema_units.png)

However, units exist as standard resources that can be reused and integrated into ontologies an vocabularies like WRA. 

It would be **beneficial** to take advantage of these standards and reuse them as much as posible.

## QUDT Units

One such standard is [QUDT](https://qudt.org/), an initiative to stnadardize not only unitsof meaurement but also allow establishing conversion paths among units and identifying equivalent quantity kinds.

The QUDT Units use a standard ontology format, and as such can be reused in many concrete contexts. Each unit includes useful information that can be used for many purposes, like referencing standard conversion factors, common related quantity kinds, or multi-language features. For example, this is some of the information related to the Meter-per-second (m/s) unit: 

![](img/qudt-ms.png)

### URIs

All this information is retreivable through a Web URL: [`https://qudt.org/vocab/unit/M-PER-SEC`](https://qudt.org/vocab/unit/M-PER-SEC), which provide all this human-readale information.

This special URL that references information like a concept in an ontology, is called a **URI** or *Uniform Resource Identifier*. It is essentially a universal ID for a concept, and makes it very useful for referencing terms.

Moreover, we can also obtain the equivalent machine-readable information about the same concept, through this URL [`https://qudt.org/vocab/unit/M-PER-SEC.ttl`](https://qudt.org/vocab/unit/M-PER-SEC.ttl) (abridged for brevity), using a technique called URI redirection:

```
unit:M-PER-SEC
  a qudt:DerivedUnit, qudt:Unit ;
  dcterms:description """Metre per second is an SI derived unit of both speed (scalar) and velocity ..."""^^rdf:HTML ;
  qudt:applicableSystem sou:CGS ;
  qudt:applicableSystem sou:CGS-EMU ;
  qudt:applicableSystem sou:CGS-GAUSS ;
  qudt:applicableSystem sou:SI ;
  qudt:conversionMultiplier 1.0 ;
  qudt:conversionMultiplierSN 1.0E0 ;
  qudt:definedUnitOfSystem sou:SI ;
  qudt:derivedCoherentUnitOfSystem sou:SI ;
  qudt:expression "$m/s$"^^qudt:LatexString ;
  qudt:hasDimensionVector qkdv:A0E0L1I0M0H0T-1D0 ;
  qudt:hasFactorUnit [
    a qudt:FactorUnit ;
    qudt:exponent 1 ;
    qudt:hasUnit unit:M ;
  ] ;
  qudt:hasFactorUnit [
    a qudt:FactorUnit ;
    qudt:exponent -1 ;
    qudt:hasUnit unit:SEC ;
  ] ;
  qudt:hasQuantityKind quantitykind:ElectromagneticWavePhaseSpeed ;
  qudt:hasQuantityKind quantitykind:LinearVelocity ;
  qudt:hasQuantityKind quantitykind:Speed ;
  qudt:hasQuantityKind quantitykind:Velocity ;
  qudt:hasReciprocalUnit unit:SEC-PER-M ;
  qudt:iec61360Code "0112/2///62720#UAA733" ;
 ...
```

### Referencing QUDT in WRA

Having these standard units and their universal identifiers, i.e. **URIs**, it seems reasonable to try to reuse these units in WRA, given that they can provide us **semanticall richer** information. 

The simples way to do so could simply be to put the **URI** of the unit instead of `"m/s"`, therefore:

```
  "measurement_units_id": "https://qudt.org/vocab/unit/M-PER-SEC"

```

It has the direct advantage of reusing a well known standard for units and not needing to reinvent the wheel.

## JSON-LD and WRA

Moreover, fortunately there is an even easier way to use these ontology terms in JSON, using a JSON dialect standardized exactly for this kind of use-case. This dilaect is [JSON-LD](https://www.w3.org/TR/json-ld/). JSON-LD is a specific serialization format basedon JSON for Linked Data, and is widely used to represent semantic information.

Essentially it allows referencing terms from ontologies by adding a `@context` object in the JSON document. You can see a JSON-LD version of our WRA example in this document: [data/simple-wra-ld.json](data/simple-wra-ld.json). 

As you can see it is still a JSON document with the same structure, but adding a `@context` object.


### JSON-LD @context

As you can see it is still a JSON document with the same structure, but adding a `@context` object.

Among other things, this context object contains a reference to the QUDT ontology through a `prefix`:

```
    "qudt": "http://qudt.org/vocab/unit/",
```

### Referencig the units

And finally you can see how the unit is referenced later on in the WRA body:

```
    "measurement_units_id": "qudt:M-PER-SEC"
```

## Representing WRA Units with triples

The key way of thinking about data in the world of ontologies is a **Graphs**. All informtation is modelled as graphs whose core is a **triple**, i.e., a *statement* composed of a (*subject*, *predicate*, *object*).

For instance, if we want to say that `M-PER_SEC` is a unit, we can have a statement like:

```
     (M-PER-SEC,type,Unit)
```

Or if we want to say that `M-PER-SEC` is defined in the International System of Units (SI), we can have this triple:

```
     (M-PER-SEC,definedUnitOfSystem,SI)
```


This is exactly the type of information that QUDT holds, and that now our enhanced WRA is capable to reference.


### Working with rdflib

Now that we are using JSON-LD, we can use libraries that are specialized in working with semantic information to exploit them and make the most of them.

One of these libraries is `rdflib`. There are others, but for the sake of these examples we will use this one. 

First we can import `rdflib`, and parse our WRA file:

In [67]:
from rdflib import Graph

g = Graph()
g.parse("data/simple-wra-ld.json")

<Graph identifier=N975d4181afe14dadb113e7d65dca9eed (<class 'rdflib.graph.Graph'>)>

### Navigating unit details

All this file contents can be explored as a graph, and within these contents is our unit of measurement: `M-PER-SEC`.

We can simply retrieve the QUDT details of this unit through the same process, by parsing this unit and all of its contents as a graph:

In [68]:
g2 = Graph()
g2.parse("https://qudt.org/vocab/unit/M-PER-SEC")

<Graph identifier=N9d89cde761cf4173b3fec4433476d260 (<class 'rdflib.graph.Graph'>)>

For example, we can see what are the labels (or translations) of this unit in multiple languages:

In [69]:
from rdflib import URIRef,Literal,Namespace
from rdflib.namespace import RDFS

UNIT=Namespace("http://qudt.org/vocab/unit/")
QUDT=Namespace("http://qudt.org/schema/qudt/")

for (s,p,o) in g2.triples((UNIT["M-PER-SEC"],RDFS.label,None)):
    if o.language == 'fr' :
        print('Unit in french is:',o)
    elif o.language == 'de' :
        print('Unit in german is:',o)


Unit in german is: Meter pro Sekunde
Unit in french is: Mètre par Seconde


We can also see for which quantities this unit is used for:

In [70]:
for (s,p,o) in g2.triples((UNIT["M-PER-SEC"],QUDT.hasQuantityKind,None)):
    print(o)
    g3 = Graph()
    g3.parse(o)
    for (s1,p1,o1) in g3.triples((o,RDFS.label,None)):
        if o1.language == 'en' : print(o1)


http://qudt.org/vocab/quantitykind/ElectromagneticWavePhaseSpeed
Electromagnetic Wave Phase Speed
http://qudt.org/vocab/quantitykind/LinearVelocity
Linear Velocity
http://qudt.org/vocab/quantitykind/Speed
Speed
http://qudt.org/vocab/quantitykind/Velocity
velocity


We can also get other useful information like the UCUM code for this unit:

In [71]:
for (s,p,o) in g2.triples((UNIT["M-PER-SEC"],QUDT.ucumCode,None)):
    print(o)

m.s-1


Or the UNECE Common code:

In [72]:
for (s,p,o) in g2.triples((UNIT["M-PER-SEC"],QUDT.uneceCommonCode,None)):
    print(o)

MTS


### Conversion of units

Moreover, we can perform standard conversions using libraries that directly use the QUDT conversion factors.

For example, consider you have a distance in meters:

In [73]:
from qudt.ontology.unit_factory import UnitFactory

m = UnitFactory.get_unit('http://qudt.org/vocab/unit#Meter')

dist = Quantity(20, m)
dist

20 m

We can transform it to another unit of measurement:

In [74]:
f = UnitFactory.get_unit('http://qudt.org/vocab/unit#Foot')
dist.convert_to(f)

65.61679790026247 ft

And we can see that it uses the ontology to check if conversions are possible, else errors are thrown saying that the units arenot compatible for conversion:

In [75]:
k = UnitFactory.get_unit('http://qudt.org/vocab/unit#Kelvin')
dist.convert_to(k)

ValueError: The new unit does not have the same parent type (source: http://qudt.org/schema/qudt#LengthUnit; target: http://qudt.org/schema/qudt#TemperatureUnit)